# Part 6 — Final Deployment and Reflection
### Sydney Property Sale Price Predictor — Streamlit App
It does **not** depend on any files produced by the main notebook — running it top to
bottom from a clean folder (with just the spreadsheet in `data/`) will reproduce
`app/model.pkl` and `app/app.py` from scratch.

**Folder layout this notebook expects / creates:**
```
your-project-folder/
├── Part6_Deployment_App.ipynb   <- this notebook
├── data/
│   └── Sydney_Property_Sales_Collected.xlsx
└── app/                          <- created automatically
    ├── model.pkl                 <- created by Cell 2
    └── app.py                    <- created by Cell 3
```


## 0. Setup

Run this once. It creates the `data/` and `app/` folders next to this notebook (if they
don't already exist) and checks the spreadsheet is where it needs to be.


In [1]:
import os

os.makedirs("data", exist_ok=True)
os.makedirs("app", exist_ok=True)

expected_path = r"C:\Users\Rohan\sydney prop\data\Sydney_Property_Sales_Collected.xlsx"
if os.path.exists(expected_path):
    print(f"Found dataset at: {os.path.abspath(expected_path)}")
else:
    print(f"Dataset NOT found at: {os.path.abspath(expected_path)}")


Found dataset at: C:\Users\Rohan\sydney prop\data\Sydney_Property_Sales_Collected.xlsx


## 1. Train the final model and export it

This repeats the same feature engineering used in Parts 2–3 of the main notebook, then
retrains the **Random Forest Regressor** (the model recommended in Part 3) on the **full
120-property dataset** — for a deployed model there is no need to hold out a test set,
since evaluation was already done in the main notebook.

The fitted pipeline, the exact feature-name lists it expects, and small lookup lists for
the app's dropdown menus are bundled together and pickled to `app/model.pkl`.


In [2]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE = 42

# ---------------------------------------------------------------------------
# 1. Load the raw collected dataset
# ---------------------------------------------------------------------------

# CORRECT DATASET PATH
DATA_PATH = r"C:\Users\Rohan\sydney prop\data\Sydney_Property_Sales_Collected.xlsx"

# Check that the file exists before loading
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found at: {DATA_PATH}")

df = pd.read_excel(
    DATA_PATH,
    sheet_name="Collected_Data"
)

df["sale_date"] = pd.to_datetime(df["sale_date"])

print(f"Loaded {len(df)} properties from:")
print(DATA_PATH)
print("Shape:", df.shape)

display(df.head())


# ---------------------------------------------------------------------------
# 2. Feature engineering
# ---------------------------------------------------------------------------

empty_cols = [c for c in df.columns if df[c].isna().all()]
df = df.drop(columns=empty_cols)

type_counts = df["property_type"].value_counts()

rare_types = type_counts[type_counts < 5].index.tolist()

df["property_type_grouped"] = df["property_type"].where(
    ~df["property_type"].isin(rare_types),
    "Other"
)

df["land_size_missing"] = df["land_size_m2"].isna().astype(int)
df["car_spaces_missing"] = df["car_spaces"].isna().astype(int)

df["land_size_m2_imp"] = (
    df.groupby("suburb")["land_size_m2"]
      .transform(lambda s: s.fillna(s.median()))
)

df["car_spaces_imp"] = (
    df.groupby("suburb")["car_spaces"]
      .transform(lambda s: s.fillna(s.median()))
)

df["days_since_first_sale"] = (
    df["sale_date"] - df["sale_date"].min()
).dt.days

df["sale_month"] = df["sale_date"].dt.month

df["total_rooms"] = (
    df["bedrooms"] + df["bathrooms"]
)

print("Feature engineering complete.")
print("Shape:", df.shape)


# ---------------------------------------------------------------------------
# 3. Train final Random Forest model
# ---------------------------------------------------------------------------

num_features = [
    "bedrooms",
    "bathrooms",
    "car_spaces_imp",
    "land_size_m2_imp",
    "land_size_missing",
    "car_spaces_missing",
    "total_rooms",
    "days_since_first_sale",
    "sale_month"
]

cat_features = [
    "suburb",
    "property_type_grouped"
]

X = df[num_features + cat_features]
y = df["sale_price_aud"]

preprocess = ColumnTransformer([
    (
        "num",
        StandardScaler(),
        num_features
    ),
    (
        "cat",
        OneHotEncoder(handle_unknown="ignore"),
        cat_features
    ),
])

final_model = Pipeline([
    ("prep", preprocess),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=3,
            random_state=RANDOM_STATE
        )
    ),
])

final_model.fit(X, y)

print("Final model trained on", len(df), "records.")

print(
    "In-sample R^2 (reference only, not a generalisation estimate):",
    round(final_model.score(X, y), 3)
)


# ---------------------------------------------------------------------------
# 4. Save model.pkl into the CORRECT app folder
# ---------------------------------------------------------------------------

APP_FOLDER = r"C:\Users\Rohan\sydney prop\app"

os.makedirs(APP_FOLDER, exist_ok=True)

MODEL_PATH = os.path.join(
    APP_FOLDER,
    "model.pkl"
)

with open(MODEL_PATH, "wb") as f:
    pickle.dump(
        {
            "model": final_model,
            "num_features": num_features,
            "cat_features": cat_features,
            "reference_date": df["sale_date"].min(),
            "suburb_options": sorted(
                df["suburb"].unique().tolist()
            ),
            "type_options": sorted(
                df["property_type_grouped"].unique().tolist()
            ),
        },
        f
    )

print("\nSaved trained model bundle to:")
print(MODEL_PATH)

print("\nModel exists:", os.path.exists(MODEL_PATH))
print("Keep model.pkl and app.py together.")

Loaded 120 properties from:
C:\Users\Rohan\sydney prop\data\Sydney_Property_Sales_Collected.xlsx
Shape: (120, 23)


,id,suburb,address,sale_price_aud,sale_date,property_type,bedrooms,bathrooms,car_spaces,land_size_m2,...,has_pool,has_garden,has_air_conditioning,has_study,has_dishwasher,has_solar,property_condition,description,sale_method,source
0,1,Mosman,22A Upper Avenue,3080000,2026-08-10,House,4,3,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Domain Mosman sold listings, page 1"
1,2,Mosman,77 Belmont Road,4900000,2026-08-05,House,4,3,1.0,462.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Domain Mosman sold listings, page 1"
2,3,Mosman,142/15 Hale Road,3800000,2026-07-24,Retirement Living,3,2,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Domain Mosman sold listings, page 1"
3,4,Mosman,5 Botanic Road,8700000,2026-07-22,House,4,3,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Domain Mosman sold listings, page 1"
4,5,Mosman,23 Dalton Road,2900000,2026-07-21,Semi-detached,3,2,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Domain Mosman sold listings, page 1"


Feature engineering complete.
Shape: (120, 19)
Final model trained on 120 records.
In-sample R^2 (reference only, not a generalisation estimate): 0.876

Saved trained model bundle to:
C:\Users\Rohan\sydney prop\app\model.pkl

Model exists: True
Keep model.pkl and app.py together.


## 2. Write the Streamlit application file

Rather than asking you to copy-paste code into a separate `app.py` file by hand (an easy
place for path mistakes to creep in), this cell **writes `app/app.py` directly**, so it is
always created in exactly the same folder as `model.pkl` from the cell above.


In [3]:
import os

os.makedirs("app", exist_ok=True)

APP_CODE = r'''"""
Sydney Property Sale Price Predictor
-------------------------------------
Streamlit web app that loads the trained Random Forest pipeline (model.pkl)
and lets a user enter property characteristics to get a predicted sale price.

Run locally with:
    pip install streamlit scikit-learn pandas
    streamlit run app.py

IMPORTANT: model.pkl must sit in the SAME FOLDER as this file.
"""

import pickle
import os
from datetime import date

import pandas as pd
import streamlit as st

st.set_page_config(page_title="Sydney Property Price Predictor", page_icon="🏠", layout="centered")

MODEL_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "model.pkl")


@st.cache_resource
def load_model(path=MODEL_PATH):
    with open(path, "rb") as f:
        return pickle.load(f)


bundle = load_model()
model = bundle["model"]
num_features = bundle["num_features"]
cat_features = bundle["cat_features"]
reference_date = bundle["reference_date"]
suburb_options = bundle["suburb_options"]
type_options = bundle["type_options"]

st.title("🏠 Sydney Property Sale Price Predictor")
st.write(
    "Enter a property's characteristics below to get an estimated sale price from a "
    "Random Forest model trained on 120 manually-collected sold listings from "
    "Mosman, Parramatta and Campbelltown."
)

with st.form("property_form"):
    col1, col2 = st.columns(2)
    with col1:
        suburb = st.selectbox("Suburb", suburb_options)
        property_type = st.selectbox("Property type", type_options)
        bedrooms = st.number_input("Bedrooms", min_value=1, max_value=10, value=3, step=1)
        bathrooms = st.number_input("Bathrooms", min_value=1, max_value=8, value=2, step=1)
    with col2:
        car_spaces = st.number_input("Car spaces", min_value=0, max_value=6, value=1, step=1)
        land_size_known = st.checkbox("Land size known?", value=True)
        land_size = st.number_input("Land size (m²)", min_value=0, max_value=5000, value=500, step=10,
                                     disabled=not land_size_known)
        sale_date = st.date_input("Sale / valuation date", value=date.today())

    submitted = st.form_submit_button("Predict sale price")

if submitted:
    land_size_missing = 0 if land_size_known else 1
    land_size_value = land_size if land_size_known else 0

    row = pd.DataFrame([{
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "car_spaces_imp": car_spaces,
        "land_size_m2_imp": land_size_value,
        "land_size_missing": land_size_missing,
        "car_spaces_missing": 0,
        "total_rooms": bedrooms + bathrooms,
        "days_since_first_sale": (pd.Timestamp(sale_date) - reference_date).days,
        "sale_month": sale_date.month,
        "suburb": suburb,
        "property_type_grouped": property_type,
    }])

    if land_size_missing == 1:
        st.info("Land size not provided — using the missing-value indicator feature; "
                 "prediction relies more heavily on suburb, bedrooms and bathrooms.")

    pred = model.predict(row[num_features + cat_features])[0]
    st.success(f"### Predicted sale price: ${pred:,.0f} AUD")
    st.caption(
        "This estimate comes from a model trained on only 120 properties across three "
        "suburbs and should be treated as an indicative starting point, not a formal "
        "valuation. See the accompanying report for model limitations."
    )

st.divider()
st.caption(
    "Model: Random Forest Regressor (scikit-learn) · Training data: 120 Domain.com.au "
    "sold listings, Mosman / Parramatta / Campbelltown · For educational use only."
)
'''

with open("app/app.py", "w", encoding="utf-8") as f:
    f.write(APP_CODE)

print("Wrote app/app.py")
print("Contents of the 'app' folder:")
for fname in sorted(os.listdir("app")):
    print(" -", fname)


Wrote app/app.py
Contents of the 'app' folder:
 - app.py
 - model.pkl


## 3. Verify the exported model works before deploying

Before launching the actual web app, it's worth loading `model.pkl` back and running one
prediction directly in the notebook — this confirms the pickle is valid and the feature
names line up, so any problems are caught here rather than inside Streamlit.


In [4]:
import pickle
import pandas as pd

with open("app/model.pkl", "rb") as f:
    bundle = pickle.load(f)

model = bundle["model"]
num_features = bundle["num_features"]
cat_features = bundle["cat_features"]

# A sanity-check prediction using a typical Parramatta house
test_row = pd.DataFrame([{
    "bedrooms": 3,
    "bathrooms": 2,
    "car_spaces_imp": 1,
    "land_size_m2_imp": 500,
    "land_size_missing": 0,
    "car_spaces_missing": 0,
    "total_rooms": 5,
    "days_since_first_sale": 300,
    "sale_month": 6,
    "suburb": "Parramatta",
    "property_type_grouped": "House",
}])

pred = model.predict(test_row[num_features + cat_features])[0]
print(f"Sanity-check prediction (3bd/2ba Parramatta house): ${pred:,.0f} AUD")
print("\nIf this ran without errors, model.pkl is valid and app.py will be able to load it.")


Sanity-check prediction (3bd/2ba Parramatta house): $1,630,840 AUD

If this ran without errors, model.pkl is valid and app.py will be able to load it.


## 4. How the application was developed

1. `model.pkl` (Cell 2) packages the fitted `scikit-learn` `Pipeline` — preprocessing
   (`StandardScaler` + `OneHotEncoder`) followed by a `RandomForestRegressor` — together
   with the exact feature-name lists the pipeline expects and small lookup lists (suburb
   options, property-type options, the reference date used to compute
   `days_since_first_sale`) so the app's dropdowns always match what the model saw during
   training.
2. `app.py` (Cell 3) loads `model.pkl` once, cached with `st.cache_resource` so it is not
   reloaded on every interaction. It renders a form (`st.selectbox`, `st.number_input`,
   `st.date_input`), reconstructs a single-row `DataFrame` with exactly the columns the
   pipeline expects, and calls `model.predict(...)` when the form is submitted.
3. **Streamlit** was chosen over Flask/Gradio because a full form-to-prediction UI needs
   no separate HTML/CSS/JS or manual routing — it is expressed in under 100 lines of pure
   Python, which suits a small, single-model educational deployment.

## 5. Instructions to build, run and use the application

```bash
# 1. app/model.pkl and app/app.py both exist (run Cells 1-3 above first).

# 2. Install the required packages:
pip install streamlit scikit-learn pandas

# 3. Launch the app -- this works from ANY starting folder, because app.py resolves
streamlit run app/app.py

# Streamlit will print a local URL (typically http://localhost:8501)
```

**Using the app:** select a suburb and property type, enter bedrooms/bathrooms/car
spaces, either enter a land size or untick "Land size known?" if it is not available,
choose a sale/valuation date, and click **Predict sale price**. The predicted price is
shown along with a plain-language caveat about the model's limitations.

